<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/03_sessions_state.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 03 — Sessions, State, Events, Artifacts

> **Where you are** — you know agents + tools (M01–M02).
> - **In M01–M02:** our `chat()` helper deliberately opened a *fresh* conversation for every message — nothing carried over, and for those demos that was fine.
> - **New in this module:** we stop throwing conversations away. You'll watch them being stored and resumed, meet the **state** dict that rides along, the prefixes that decide what survives, and events as the record of everything.
> - **No new Python** — one new ADK idea: ADK *injecting* an argument into your tool (explained when we get there).

If Module 02 was about what an agent can *do*, Module 03 is about what an agent *remembers*.

Start with the everyday problem. A user talks to your agent today and comes back tomorrow. For the agent to feel like a colleague and not a goldfish, something has to be **stored** in the meantime — and that one word immediately raises very practical questions:

- **Where?** In RAM? In a file? In a database?
- **What exactly?** The whole conversation word for word — or just the useful facts pulled out of it, like *favorite color: teal*?
- **For whom?** Should tomorrow's conversation see it? Should *other users'* conversations?

Here is the good news: for the conversation itself the answer already exists — the **Session** stores the whole dialogue. You just haven't *felt* it yet, because in M01–M02 our helper opened a fresh session for every single message. So that is where we start: a demo where conversations are stored, left, and picked up again. Then we open the box, add the second kind of memory (the extracted facts, kept in **state**), and learn the five-character trick that answers *"for whom, and for how long"*.

**What we'll do:**

1. **See it first:** hold two separate conversations with one agent, walk away, and resume either one.
2. Open the Session box: what exactly it holds.
3. Meet **state prefixes** — the naming convention that decides how long each piece of data lives.
4. The wow demo: the same user opens a **second, separate session** — and the agent still remembers them.
5. Walk the event history — the record of everything that happened.
6. The one natural-looking way to write state that silently *doesn't* persist — and why the events explain it.
7. A short word on artifacts (big files).

**Running cost:** under $0.01 on OpenRouter.

# Setup

Same ritual as every module: install, key, imports.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


Same OpenRouter key — picked up from Colab secrets or `.env`.

In [2]:
import os

OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Set OPENROUTER_API_KEY in Colab secrets (🔑 icon) or a local .env file.")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


One import is new: `ToolContext` — it appears in the favorite-color demo later in the module.

In [3]:
import os
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try:
    sys.stderr.fileno()
except Exception:
    sys.stderr = open(os.devnull, "w")

import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.tool_context import ToolContext
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# See It First — Conversations Are Stored, and You Can Come Back

Before any theory, let's *watch* the thing this module is about. The claim: ADK keeps whole conversations. You can hold several at once, walk away from one, and later reopen exactly the one you want — and the agent picks up where you left off.

Three steps:

1. **Monday:** we chat with a travel agent about a trip — and check it remembers, within the conversation.
2. **Tuesday:** we open a *second* conversation — and check it starts blank (conversations don't leak into each other).
3. **Back to Monday:** we reopen the first conversation — and the agent still knows everything from step 1.

### `chat()` grows one argument

The helper below is the M01 helper with one addition — and the addition is the whole module. In M01, every call quietly created a brand-new session, which is exactly why nothing ever carried over. Now `chat()` takes a third argument, `sid` — the **session id**, the label on the conversation folder:

- **You pick the label.** `chat(agent, "...", sid="monday")` lands the message in the conversation called *monday*. Call again with the same label — same conversation, continued.
- **Get or create.** The first two lines look the session up and create it only if it doesn't exist yet. The first call with a new label opens the folder; every later call reopens it.

The `# NEW` comments in the code mark exactly these spots — the rest is unchanged from M01.

In [4]:
APP = "m03_demo"
USER = "alice"
session_service = InMemorySessionService()

# A plain agent for the demo — no tools, nothing fancy.
travel_agent = LlmAgent(
    name="travel_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Helps the user plan trips.",
    instruction="You are a brief, friendly travel-planning assistant. "
                "If you don't know something, say so plainly instead of guessing.",
)

async def chat(agent, prompt: str, sid: str):
    # NEW vs M01: `sid` — WE choose which conversation this message belongs to.
    sess = await session_service.get_session(app_name=APP, user_id=USER, session_id=sid)
    if sess is None:
        # NEW: create the session only if it doesn't exist yet ("get or create").
        await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER ({sid}): {prompt}")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.thought:  # the model's private reasoning summary (you saw it in M02) — skip it here
                    continue
                if p.text and p.text.strip():
                    tag = "[FINAL]" if event.is_final_response() else "[step]"
                    print(f"{tag} {event.author}: {p.text.strip()[:200]}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args)})")
                if p.function_response:
                    print(f"[tool_resp] {p.function_response.response}")
    print()

print("✅ travel_agent and chat() ready.")

✅ travel_agent and chat() ready.


In [5]:
# Step 1 — Monday: one conversation, two messages.
await chat(travel_agent, "Hi! I'm planning a trip to Vienna in September.", sid="monday")
await chat(travel_agent, "Which city am I going to, again?", sid="monday")

USER (monday): Hi! I'm planning a trip to Vienna in September.


[FINAL] travel_agent: Hi! Vienna in September is a lovely choice—generally mild, with comfortable sightseeing weather and the start of autumn.

I can help plan your trip. Do you already know:
- Your travel dates and number

USER (monday): Which city am I going to, again?


[FINAL] travel_agent: You’re planning a trip to **Vienna, Austria**.



### 🔍 What just happened?

The second answer is the point: the agent knew *Vienna*, even though your second message never mentioned it. It could know because both messages carry the same label — `sid="monday"` — so they belong to one stored conversation, and on every turn the Runner hands the model the **whole conversation so far**, not just the newest message. In the previous course you did this by hand, appending to a `messages` list. A session is that list, with a proper home.

In [6]:
# Step 2 — Tuesday: a brand-new conversation. Same question, different label.
await chat(travel_agent, "Which city am I going to, again?", sid="tuesday")

# Both conversations now sit in the session service — list them.
resp = await session_service.list_sessions(app_name=APP, user_id=USER)
print("Stored conversations:", sorted(s.id for s in resp.sessions))

USER (tuesday): Which city am I going to, again?


[FINAL] travel_agent: I don’t have your destination in this chat. Which trip are you referring to?

Stored conversations: ['monday', 'tuesday']


In [7]:
# Step 3 — back to Monday: reopen the first conversation and continue it.
await chat(travel_agent, "One more thing about my trip — remind me which city it was?", sid="monday")

USER (monday): One more thing about my trip — remind me which city it was?


[FINAL] travel_agent: Vienna, Austria.



### 🔍 What just happened?

In *tuesday* the agent honestly had no idea — that conversation starts from zero, because a fresh label means a fresh folder. Yet nothing was lost: the listing shows both conversations stored side by side, and when we reopened *monday*, the agent answered *Vienna* again. Leave, come back, continue — that is what sessions give you before you write a single line of "memory code".

### 🎯 Mini-task

Open a third conversation `sid="weekend"` and plan a different trip in it. Then continue *tuesday*. And then the interesting one: **restart the notebook kernel**, re-run the cells, and list the conversations again. What survived? Keep your answer in mind — the next section explains it.

---

So conversations are stored and resumable. Time to open the box: what exactly does a session hold?

# What a Session Holds

The `Session` you just watched at work, one level deeper. ADK files every conversation under a triple: `(app_name, user_id, session_id)` — which app, which user, which of their conversations. Inside, a session holds two things:

- **A list of events** — the full ordered history: every message, tool call, tool response. ADK appends to it; your code can read it.
- **A state dict** — an ordinary key-value store for the *extracted facts*: `{"favorite_color": "teal"}`. This dict is the module's main character.

And where does all of this physically live? So far: **in RAM.** `InMemorySessionService` is literally a Python dict inside your notebook process — close the notebook and everything is gone. That is deliberately naive, and perfectly fine for learning. In M08 we swap it for a real database *with one changed line*, and nothing else you learn today changes — so storage is not our worry yet.

# State Prefixes — Who Remembers What, and For How Long

Say your agent learns the user's favorite color. Should it still know it tomorrow, in a new conversation? Should *other* users' agents know it? Not the same answer for every piece of data — and ADK's answer is a naming convention:

**The key's prefix decides how far a value travels and how long it survives.**

| Prefix | Scope | Survives |
|---|---|---|
| *(none)* | This session only | Until the session is deleted |
| `user:` | This user, across all their sessions | As long as the user exists |
| `app:` | Global across all users of this app | As long as the app exists |
| `temp:` | The current turn only | Thrown away once the reply is done |

Four rings, from throwaway scratch paper to global settings, chosen with a few characters at the front of a dict key. The demo uses the two you'll need most: no prefix and `user:`.

# Writing State From a Tool

We want the agent to **remember the user's favorite color**. How does the color get into the state dict? With a tool — an ordinary tool, exactly like the ones you built in M02:

```python
def remember_favorite_color(color: str, tool_context: ToolContext) -> dict:
    tool_context.state["user:favorite_color"] = color
```

The model does what it always does with tools: it hears *"my favorite color is teal"*, decides this tool fits, extracts `color='teal'`, and calls it. Inside, the function writes the value into state — a plain dict assignment, with the `user:` prefix so it survives into future sessions. And recall is the same trick in the other direction: a second tool reads the dict and returns the value to the model.

### Two things in the next cell that nobody told you about yet

**1. The strange second parameter: `tool_context`.** Look at the function again — the model only supplies `color`. So where does `tool_context` come from? **ADK fills it in.** When ADK sees a parameter typed `ToolContext` in a tool's signature, it reads it as a note addressed to itself: *"when you run this tool, hand it the running session here."* The model never even learns this parameter exists — it isn't in the schema the model sees. What you get from it: inside a tool, `tool_context.state` **is** the session's state dict. Read it, write it — you are reading and writing the conversation's memory.

**2. One extra argument on the agent: `output_key="last_response"`.** A convenience switch. It tells ADK: *after every reply this agent gives, save the reply's final text into state under the key `last_response`.* There is nothing to call — it simply happens, every turn. We add it so the demo shows a second, different way state gets written.

So when you run the demo, watch for **two** writes into state: one made by the tool (key `user:favorite_color`), one made automatically after the reply (key `last_response`).

In [8]:
def remember_favorite_color(color: str, tool_context: ToolContext) -> dict:
    """Record the user's favorite color for future sessions.

    Args:
        color: A color name like "teal", "crimson", "forest green".
    """
    # Note the user: prefix — this makes the value survive beyond this session.
    tool_context.state["user:favorite_color"] = color
    return {"status": "saved", "color": color}


def recall_favorite_color(tool_context: ToolContext) -> dict:
    """Check whether the user has told us their favorite color before."""
    color = tool_context.state.get("user:favorite_color")
    if color:
        return {"known": True, "color": color}
    return {"known": False}


color_agent = LlmAgent(
    name="color_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Remembers and recalls the user's favorite color across sessions.",
    instruction=(
        "You track the user's favorite color. "
        "When they tell you a color, call remember_favorite_color. "
        "When they ask what their color is, call recall_favorite_color first. "
        "Be brief."
    ),
    tools=[remember_favorite_color, recall_favorite_color],
    output_key="last_response",   # also save the final text to state['last_response']
)

print("✅ color_agent ready.")

✅ color_agent ready.


## Run It — Session 1

Same `chat()` helper as in the opening demo — sessions are just labels, so the color agent gets its own: `"session-one"`. At the end of the cell we fetch the session back and print its state — our first look inside the dict.

In [9]:
SID_1 = "session-one"
await chat(color_agent, "My favorite color is teal.", sid=SID_1)

# Inspect what's in the session's state now.
s1 = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_1)
print("── Session 1 state after the conversation ──")
for k, v in sorted(dict(s1.state).items()):
    print(f"  {k!r}: {v!r}")

USER (session-one): My favorite color is teal.


[tool_call] remember_favorite_color({'color': 'teal'})
[tool_resp] {'status': 'saved', 'color': 'teal'}


[FINAL] color_agent: Got it!

── Session 1 state after the conversation ──
  'last_response': 'Got it!'
  'user:favorite_color': 'teal'


### 🔍 What just happened?

Two separate writes landed in state:

- The tool wrote `user:favorite_color = 'teal'` — you can see the `[tool_call]` in the stream.
- `output_key="last_response"` auto-saved the model's final sentence under `last_response`.

Both are now part of the session. Now the real test: a **completely separate session** for the same user. The `user:`-prefixed key should carry over. The unprefixed `last_response` should not.

## Session 2 — Same User, Fresh Session

This is the moment the module exists for. We create session two with no initial state at all, print what it starts with, and then ask the agent a question it can only answer if the prefix system works.

In [10]:
SID_2 = "session-two"

# Create the second session fresh — notice we don't pass any initial state.
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID_2)
s2_initial = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)

print("── Session 2 initial state (before any chat) ──")
for k, v in sorted(dict(s2_initial.state).items()):
    print(f"  {k!r}: {v!r}")
print()

# Ask the agent in session 2 — does it remember?
await chat(color_agent, "Hey, what's my favorite color?", sid=SID_2)

── Session 2 initial state (before any chat) ──
  'user:favorite_color': 'teal'

USER (session-two): Hey, what's my favorite color?


[tool_call] recall_favorite_color({})
[tool_resp] {'known': True, 'color': 'teal'}


[FINAL] color_agent: Your favorite color is teal.



### 🔍 What just happened?

Read the output top to bottom:

- Session 2 **started** with `user:favorite_color: 'teal'` already present — it survived.
- `last_response` from session 1 is **not** there — unprefixed, so it stayed behind.
- The agent called `recall_favorite_color`, read the surviving key, and answered "teal".

That is cross-session memory in 80 lines of code. No database, no vector store — a prefix convention on a dict key. You decide the lifetime of every piece of data with five characters, and ADK handles the rest.

### 🎯 Mini-tasks

1. **App scope.** Write a tool `set_app_mode(mode: str, tool_context: ToolContext)` that stores `tool_context.state["app:mode"] = mode`. Create two different users (change `USER` between calls). Does `app:mode` show up for both?
2. **Temp scope.** Write a tool that stores `tool_context.state["temp:scratch"] = "..."`. After the run completes, fetch the session and inspect the state. Is `temp:scratch` still there?

# Events — Everything That Happened, In Order

The session's other half is the **event list** — and unlike state, you never write it yourself; ADK has been quietly appending to it all along. Why would you go back and *read* it? Three everyday reasons:

- A user says *"the agent told me something wrong yesterday"* — you replay the conversation and see exactly which tool returned what.
- You want a clean transcript of a conversation — you rebuild it from the events.
- You want to test the agent — the events are the record you check against (M09 does exactly this).

The one-line version: **a session's event list is the agent's log file, with structure.** The next cell walks session 1's history and labels each event.

In [11]:
# Walk the event history of session 1 — one line per event.
s1 = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_1)
print(f"Session 1 has {len(s1.events)} events.\n")

for ev in s1.events:
    what = []
    for p in (ev.content.parts if ev.content else []):
        if p.thought:
            continue
        if p.text and p.text.strip():
            what.append("text: " + p.text.strip()[:40])
        if p.function_call:
            what.append("tool_call: " + p.function_call.name)
        if p.function_response:
            what.append("tool_resp: " + p.function_response.name)
    if ev.actions.state_delta:
        what.append("state_delta: " + str(list(ev.actions.state_delta)))
    print(ev.author.ljust(12), " | ".join(what))

Session 1 has 4 events.

user         text: My favorite color is teal.
color_agent  tool_call: remember_favorite_color
color_agent  tool_resp: remember_favorite_color | state_delta: ['user:favorite_color']
color_agent  text: Got it! | state_delta: ['last_response']


### 🔍 What just happened?

Notice the `state_delta` entries. Every state write — the tool's, `output_key`'s, session setup — was recorded as an event. The state dict you inspected earlier is not stored separately: **ADK rebuilds it by replaying these deltas in order** whenever you fetch the session.

This design has a name — [event sourcing](https://martinfowler.com/eaaDev/EventSourcing.html): the history is the truth, and the current state is computed from it. Banks and databases work this way; ADK does the same for conversations. Hold on to this picture — the next section shows the one natural-looking way of writing state that breaks exactly this rule.

### 🎯 Mini-task

Write a loop that walks `s1.events` and prints only the user's and the agent's *text* messages, in order — a clean, readable transcript with none of the tool machinery.

# Three Ways to Write State — Only Two Persist

So far state has been written in two ways, and both stuck:

1. **From inside a tool:** `tool_context.state["user:favorite_color"] = "teal"`.
2. **Automatically, after every reply:** `output_key="last_response"` saved each reply's final text — you watched the key appear without anyone calling anything.

There is a third way that *looks* the most natural of all: fetch the session and assign into its state directly — `session.state["user:mood"] = "happy"`. **It silently does not persist.** No error, no warning; the next fetch simply returns the old state.

You already know why. State lives in the event list: the two working ways each record a `state_delta` event, and the dict you see is rebuilt from those. Direct assignment writes into a copy in your hands and records no event — so there is nothing to replay.

The rule: **write state from a tool, or via `output_key`. Never assign into a fetched session and expect it to stick.** The next cell walks into the trap on purpose, so you recognise it later in your own code.

In [12]:
# Demonstrate the pitfall — do NOT copy this pattern.
# Mutate state directly and re-fetch to prove it didn't stick.
sess = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)
sess.state["user:mood"] = "happy"                 # direct assignment — WRONG
sess.state["this_does_not_persist"] = "ghost"     # unprefixed — also wrong, but doubly so

sess_fresh = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)
print("── Re-fetched session state ──")
for k, v in sorted(dict(sess_fresh.state).items()):
    print(f"  {k!r}: {v!r}")
print()
print("Notice: 'user:mood' and 'this_does_not_persist' are NOT in the re-fetched state.")
print("Direct assignment to a returned session's .state does not round-trip.")

── Re-fetched session state ──
  'last_response': 'Your favorite color is teal.'
  'user:favorite_color': 'teal'

Notice: 'user:mood' and 'this_does_not_persist' are NOT in the re-fetched state.
Direct assignment to a returned session's .state does not round-trip.


### 🔍 What just happened?

Both direct assignments vanished on re-fetch — `user:mood` *and* the unprefixed key. The prefix didn't matter: no event was recorded, so nothing persisted. When state "mysteriously doesn't save" in your own agent, this is the first thing to check.

# Artifacts — Big Files, Briefly

One more drawer in the session, mentioned so you know it exists: **artifacts** are for binary data — images, audio, PDFs — that belong to a conversation but shouldn't be stuffed into the event list. The event stream keeps a small pointer; the file itself lives in a separate store (`InMemoryArtifactService` for learning, Google Cloud Storage or your own store in production). The text-only agents in Part 1 never need them.

# Key Takeaways

- A **Session** is a triple `(app_name, user_id, session_id)` holding events and a state dict.
- **State prefixes** decide lifetime: `user:` survives across sessions, `app:` is global, `temp:` is per-turn, unprefixed is per-session.
- **Events are the record of everything, in order.** State is rebuilt by replaying the state-delta events.
- **Only two ways to write state persist:** `output_key=` on the agent and `tool_context.state[...]` inside a tool. Direct assignment on a fetched session does NOT stick — no event, no persistence.
- **Artifacts** hold big files next to the conversation — a pointer in the events, the payload elsewhere.
- **Stored state is a hint, not a fact.** Before a high-stakes action (sending an email, charging a card), have the agent re-check with a read-only tool instead of trusting last month's memory.

# Next up — M04: The One-Line Model Swap

You've typed `LiteLlm(model="openrouter/openai/gpt-5.6-luna")` all course. M04 opens that up: the same agent on Claude, GPT, Qwen and a locally-hosted Ollama model — plus the one prefix mistake that causes infinite tool-call loops.